<a href="https://colab.research.google.com/github/tomzavi56/South-Africa-Labour-Market-Analysis/blob/main/06_regression_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Importing the relevant packages

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Connecting Google Drive to Colab

from google.colab import drive

drive.mount("/content/drive")

# Loading the QLFS Q2 2026 dataset from Google Drive

filename = "/content/drive/MyDrive/South_Africa_Labour_Market/QLFS202602.csv"

df = pd.read_csv(filename)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

# Selecting the working-age population

regression_df = df[
    (df["Q14AGE"] >= 15) &
    (df["Q14AGE"] <= 64)
].copy()


# Converting labour market status to numeric values

regression_df["Status"] = pd.to_numeric(
    regression_df["Status"],
    errors="coerce"
)


# Keeping employed and unemployed people only

regression_df = regression_df[
    regression_df["Status"].isin([1, 2])
].copy()


# Creating the binary unemployment variable

regression_df["unemployed"] = np.where(
    regression_df["Status"] == 2,
    1,
    0
)


# Selecting the variables for the regression model

regression_df = regression_df[
    [
        "unemployed",
        "Q13GENDER",
        "Q14AGE",
        "Education_status",
        "Province"
    ]
].copy()


print("Regression dataset shape:", regression_df.shape)

print("\nFirst five observations:")
display(regression_df.head())

print("\nMissing values:")
print(regression_df.isnull().sum())

# Creating readable labels for categorical variables

gender_labels = {
    1: "Male",
    2: "Female"
}

education_labels = {
    1: "No schooling",
    2: "Less than primary completed",
    3: "Primary completed",
    4: "Secondary not completed",
    5: "Secondary completed",
    6: "Tertiary",
    7: "Other"
}

province_labels = {
    1: "Western Cape",
    2: "Eastern Cape",
    3: "Northern Cape",
    4: "Free State",
    5: "KwaZulu-Natal",
    6: "North West",
    7: "Gauteng",
    8: "Mpumalanga",
    9: "Limpopo"
}

regression_df["gender_name"] = (
    regression_df["Q13GENDER"]
    .map(gender_labels)
)

regression_df["education_name"] = (
    regression_df["Education_status"]
    .map(education_labels)
)

regression_df["province_name"] = (
    regression_df["Province"]
    .map(province_labels)
)

display(
    regression_df[
        [
            "unemployed",
            "gender_name",
            "Q14AGE",
            "education_name",
            "province_name"
        ]
    ].head()
)

# Fitting the logistic regression model

model = smf.logit(
    formula=(
        "unemployed ~ C(gender_name) + Q14AGE "
        "+ C(education_name) + C(province_name)"
    ),
    data=regression_df
).fit()

print(model.summary())

# Calculating odds ratios for the logistic regression model

odds_ratios = pd.DataFrame({
    "Coefficient": model.params,
    "Odds_Ratio": np.exp(model.params),
    "P_Value": model.pvalues
})

confidence_intervals = model.conf_int()

odds_ratios["CI_Lower"] = np.exp(
    confidence_intervals[0]
)

odds_ratios["CI_Upper"] = np.exp(
    confidence_intervals[1]
)

display(
    odds_ratios.round(3)
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully.
Shape: (61977, 200)
Regression dataset shape: (22637, 5)

First five observations:


,unemployed,Q13GENDER,Q14AGE,Education_status,Province
0,0,1,29,6,1
1,0,2,28,6,1
3,0,1,49,5,1
4,0,2,48,5,1
5,0,1,62,6,1



Missing values:
unemployed          0
Q13GENDER           0
Q14AGE              0
Education_status    0
Province            0
dtype: int64


,unemployed,gender_name,Q14AGE,education_name,province_name
0,0,Male,29,Tertiary,Western Cape
1,0,Female,28,Tertiary,Western Cape
3,0,Male,49,Secondary completed,Western Cape
4,0,Female,48,Secondary completed,Western Cape
5,0,Male,62,Tertiary,Western Cape


Optimization terminated successfully.
         Current function value: 0.564918
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:             unemployed   No. Observations:                22637
Model:                          Logit   Df Residuals:                    22620
Method:                           MLE   Df Model:                           16
Date:                Wed, 14 Oct 2026   Pseudo R-squ.:                  0.1162
Time:                        07:57:08   Log-Likelihood:                -12788.
converged:                       True   LL-Null:                       -14470.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

,Coefficient,Odds_Ratio,P_Value,CI_Lower,CI_Upper
Intercept,2.660,14.293,0.000,11.479,17.797
C(gender_name)[T.Male],-0.431,0.650,0.000,0.612,0.690
C(education_name)[T.No schooling],-0.222,0.801,0.257,0.546,1.176
C(education_name)[T.Other],-0.463,0.629,0.006,0.454,0.873
C(education_name)[T.Primary completed],-0.001,0.999,0.995,0.786,1.271
C(education_name)[T.Secondary completed],-0.316,0.729,0.000,0.617,0.862
C(education_name)[T.Secondary not completed],0.093,1.097,0.272,0.930,1.295
C(education_name)[T.Tertiary],-1.053,0.349,0.000,0.292,0.417
C(province_name)[T.Free State],-0.103,0.902,0.141,0.786,1.035
C(province_name)[T.Gauteng],-0.228,0.796,0.000,0.719,0.881
